# Анализ популярности игровых платформ

## Введение

<div>
    <b>Цель проекта:</b> повысить узнаваемость видеоигры «Секреты Темнолесья» через размещение аналитического контента на известном IT-ресурсе
    <br><br>
    <b>Задачи:</b>
    <ol>
        <li>Выгрузить и изучить датасет, проверить соответствие данных описанию</li>
        <li>Провести предобработку данных:
            <ol style="list-style-type: decimal;">
                <li>Выявить количество пропусков, понять причины их возникновения, заполнить или удалить их</li>
                <li>Изучить типы данных и при необходимости изменить их для корректного использования специальных методов</li>
                <li>Определить уникальные значения в столбцах, сравнить их написание, привести названия столбцов к стилю snake_case и удалить дубликаты</li>
                <li>Отфильтровать очищенный датафрейм по периоду с 2000 по 2013 год и сохранить новый датафрейм</li>
            </ol>
        </li>
        <li>Провести исследовательский анализ данных:
            <ol style="list-style-type: decimal;">
                <li>Провести категоризацию данных по оценкам пользователей и критиков</li>
                <li>Выделить топ-7 платформ по количеству игр, выпущенных за требуемый период</li>
            </ol>
        </li>
        <li>Оформить выводы, подготовить статью о развитии мировой индустрии игр с 2000 по 2013 год на основе полученных данных и выпустить её на известном IT-ресурсе</li>
    </ol>
</div>


### Описание данных

Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
- `Name` - название игры.
- `Platform` - название платформы
- `Year of Release` - год выпуска игры.
- `Genre` - жанр игры.
- `NA sales` -  продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` — продажи в Европе (в миллионах проданных копий).
- `JP sales` — продажи в Японии (в миллионах проданных копий).
- `Other sales` — продажи в других странах (в миллионах проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

## Загрузка данных и знакомство с ними

Импортируем библиотеки `pandas`, добавляем датасета `/datasets/new_games.csv`.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/datasets/new_games.csv')

Выводим первые 5 строк датасета

In [3]:
display(df.head())

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Знакомимся с данными с помощью метода `info()`

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Выявляем процентное и количественное содержание пропусков в столбце с помощью метода isna()

In [5]:
df.isna().sum()

Name                  2
Platform              0
Year of Release     275
Genre                 2
NA sales              0
EU sales              0
JP sales              0
Other sales           0
Critic Score       8714
User Score         6804
Rating             6871
dtype: int64

In [6]:
round(df.isna().sum() / df.shape[0] * 100, 2)

Name                0.01
Platform            0.00
Year of Release     1.62
Genre               0.01
NA sales            0.00
EU sales            0.00
JP sales            0.00
Other sales         0.00
Critic Score       51.39
User Score         40.13
Rating             40.52
dtype: float64

**Выводы**:
- датасет содержит **16956** строк и **11** столбцов.
- данные представлены в двух типах: `float64` - тип чисел с плавающей точкой и **object**.
- пропуски содержатся в столбцах `Name`, `Year of Release`, `Genre`, `Critic Score`, `User Score`, `Rating`, высокое содержание пропусков в столбцах `Critic Score`(51%), `User Score`(40%), `Rating`(41%) требует заполнение пропусков, строки с пропусками в столбцах `Name` и `Genre` можно удалить.


**Анализ соответствия типов данным**

Тип `float64` может быть избыточным для хранения данных в столбце `Year of Release`, так как этот столбец содержит в себе только **целочисленное четырехзначное число**, год выпуска игры. Его можно будет заменить на тип `int16` после обработки пропусков. В числовых значениях `NaN` всегда представлен как объект типа `float64`. 

Столбцы, содержащие данные о продажах копий, а именно `NA sales`, ` EU sales`, `JP sales` и  `Other sales` представлены в двух типах данных, в `float64` и `object`, при этом в данных нет пропусков, значит, некоторые значения могли иметь тип `str.` 

Помимо этого для данных не требуется столько выделяемой памяти, так как содержатся округленные до 2 знаков значения.

## Проверка ошибок в данных и их предобработка

### Названия, или метки, столбцов датафрейма

Выводим на экран названия всех столбцов датафрейма и проверяем их стиль написания.

In [7]:
for name_column in df:
    print(name_column)

Name
Platform
Year of Release
Genre
NA sales
EU sales
JP sales
Other sales
Critic Score
User Score
Rating


Приводим все столбцы к стилю snake case, для этого названия переводим в нижний регистр, а пробелы заменяем подчеркиванием.

In [8]:
def to_snake_case (column):
    return column.lower().replace(' ', '_')

Применяем функцию и приводим столбец к стилю snake case

In [9]:
new_names = []
for column_name in df.columns:
    new_names.append(to_snake_case(column_name))
df.columns = new_names

In [10]:
for name_column in df:
    print(name_column)

name
platform
year_of_release
genre
na_sales
eu_sales
jp_sales
other_sales
critic_score
user_score
rating


### Исследование явных и неявных дубликатов в данных

Перед обработкой пропусков проверим данные на наличие явных и неявных дубликатов. Повторяющиеся записи могут приводить к завышению количества наблюдений и искажать рассчитанные показатели. Кроме того, неявные дубликаты в категориальных признаках могут возникать из-за различий в написании одного и того же значения. Поэтому сначала очистим данные от дубликатов, а затем повторно оценим количество пропусков на очищенном наборе данных.

Проверим столбцы `platform`,`genre`, `rating` и `year_of_release` содержащие категориальные значение, на **наличие неявных дубликатов.**

In [11]:
unique_platform = df['platform'].unique()
unique_platform

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

При проверке уникальных названий платформ неявных дубликатов не обнаружено. Сокращения, используемые для отдельных игровых платформ соответствуют разным поколениям, поэтому дополнительные объединения не требуются.

In [12]:
unique_genres = df['genre'].unique()
unique_genres

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan, 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION',
       'SHOOTER', 'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE',
       'SIMULATION', 'PUZZLE', 'STRATEGY'], dtype=object)

При проверке уникальных названий платформ обнаружены неявные дубликаты - жанры написаны в разным регистре. Проведем нормализацию данных для приведения названий к одному виду

In [13]:
df['genre'] = df['genre'].str.lower()

Выведем уникальные названия игровых жанров после нормализации

In [14]:
unique_genres = df['genre'].unique()
unique_genres

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy', nan], dtype=object)

In [15]:
unique_ratings = df['rating'].unique()
unique_ratings

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

В столбце `rating` обнаружено значение `K-A`, являющееся устаревшим обозначением рейтинга **ESRB.** Оно соответствует категории `E`, однако поскольку столбец `rating` не используется в дальнейшем анализе, дополнительная **нормализация** данного значения **не требуется.**

In [16]:
unique_years = df['year_of_release'].unique()
unique_years

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

Неявных дубликатов в столбце `year_of_release` не обнаружено.

Данные в столбце `name` могут отличаться наличием/отсутствием пробелов, подчеркиваний. При этом они могут относиться к одной платформе или быть адаптированной под разные платформы. Нормализуем данные и в этом столбце, чтобы в дальнейшем можно было удалить явные дубликаты с помощью встроенных методов.

In [17]:
df['name'] = df['name'].str.lower()

Найдем количество встречаемых явных дубликатов.

In [18]:
df.duplicated().sum()

241

В данных встречается **241 индентичная строка.** Удалим данные строки с помощью метода `drops_duplicates`

In [19]:
cleaned_df = df.drop_duplicates().copy()

Вернем новый датафрейм `cleaned_df`, содержащий данные **без** явных и неявных **дубликатов.** Не перезаписываем данные `df`, так как объем строк исходного датафрейма будет использоваться при подсчете количества удаленных строк в абсолютном и относительном значении.

In [20]:
cleaned_df.duplicated().sum()

0

Подсчитаем абсолютное и относительное значение удаленных пропусков

In [21]:
deleted_str = df.shape[0] - cleaned_df.shape[0]

In [22]:
deleted_str

241

Количество удаленных строк

In [23]:
round(deleted_str / df.shape[0],2) * 100

1.0

Процент удаленных данных от общего числа строк исходного датафрейма

### Обработка пропущенных значений

При знакомстве с данными мы обнаружили пропуски и подсчитали их значения в абсолютных и относительных значениях. В этой части нашего исследования мы обработаем пропуски, чтобы в дальнейшем привести данные к подходящему типу и избежать ошибок в результатах анализа.

В рамках поставленных задач используются данные столбцов `platform`, `critic_score` и `user_score`. Столбцы `name` и `rating` **не участвуют** в последующем анализе, поэтому обработка пропусков в них **не требуется**. `year_of_release` используется для ограничения анализируемого периода, поэтому пропуски в этом столбце необходимо **учитывать отдельно** при подготовке данных.

Строго определить механизм возникновения пропусков в столбцах `critic_score` и `user_score` и их вид (MCAR, MAR или MNAR) на основании имеющихся данных нельзя, поскольку для этого необходима информация о процессе формирования данных. Поэтому вместо формального присвоения типа пропусков исследуем их связь с доступными признаками и выберем способ обработки **на основании выявленных закономерностей**.

Добавим два столбца `is_critic_score_missing` и `is_user_score_missing` содержащие `True` в случае, если **не проставлена** оценка пользователя или критика, `False` **в противоположном случае.**

In [24]:
cleaned_df['is_critic_score_missing'] = cleaned_df['critic_score'].isna()

In [25]:
cleaned_df['is_user_score_missing'] = cleaned_df['user_score'].isna()

Определим, связаны ли пропуски с годом выпуска игры, жанром или платформой. А для этого подсчитаем абсолютное и относительное количество пропусков в столбцах `critic_score` и `user_score` с применением метода`groupby`.

In [26]:
platform_total_count = cleaned_df.groupby('platform')['platform'].count()

Количество игр для каждой платформы

In [27]:
platform_missing_scores_count = cleaned_df.groupby('platform')['is_critic_score_missing'].sum()

Количество пропущенных оценок критиков для каждой платформы.

In [28]:
platform_missing_scores_pct = round(cleaned_df.groupby('platform')['is_critic_score_missing'].sum() / cleaned_df.groupby('platform')['platform'].count(),2) * 100

Процент пропущенных оценок критиков для каждой платформы

In [29]:
platform_missing_df = pd.concat([platform_total_count, platform_missing_scores_count, platform_missing_scores_pct], axis=1)

Создаем датафрейм `platform_missing_df`, содержащий в себе столбцы с количеством игр на каждой платформе, количеством и процентом пропущенных оценок критиков для каждой платформы.

In [30]:
genre_total_count = cleaned_df.groupby('genre')['genre'].count()

Количество игр в каждом жанре

In [31]:
genre_missing_scores_count = cleaned_df.groupby('genre')['is_critic_score_missing'].sum()

Количество пропущенных оценок критиков для каждого жанра.

In [32]:
genre_missing_scores_pct = round(cleaned_df.groupby('genre')['is_critic_score_missing'].sum() / cleaned_df.groupby('genre')['genre'].count(),2) * 100

Процент пропущенных оценок критиков для каждого жанра.

In [33]:
genre_missing_df = pd.concat([genre_total_count, genre_missing_scores_count, genre_missing_scores_pct], axis=1)

Создаем датафрейм `genre_missing_df`, содержащий в себе столбцы с количеством игр каждого жанра, количеством и процентом пропущенных оценок критиков для каждого жанра.

In [34]:
year_total_count = cleaned_df.groupby('year_of_release')['year_of_release'].count()

Количество игр по году выпуска

In [35]:
year_missing_scores_count = cleaned_df.groupby('year_of_release')['is_critic_score_missing'].sum()

Количество пропущенных оценок критиков по каждому году.

In [36]:
year_missing_scores_pct = round(cleaned_df.groupby('year_of_release')['is_critic_score_missing'].sum() / cleaned_df.groupby('year_of_release')['year_of_release'].count(),2) * 100

Процент пропущенных оценок критиков по каждому году.

In [37]:
year_missing_df = pd.concat([year_total_count, year_missing_scores_count, year_missing_scores_pct], axis=1)

Создаем датафрейм `year_missing_df`, содержащий в себе столбцы с количеством игр для каждого года выпуска, количеством и процентом пропущенных оценок критиков для каждого года выпуска.

In [38]:
platform_missing_df.columns = ['Всего игр', 'Пропущено оценок', 'Процент пропусков']

In [39]:
platform_missing_df

,Всего игр,Пропущено оценок,Процент пропусков
platform,,,
2600,133,133,100.0
3DO,3,3,100.0
3DS,520,352,68.0
DC,52,38,73.0
DS,2151,1434,67.0
GB,98,98,100.0
GBA,822,384,47.0
GC,556,108,19.0
GEN,29,29,100.0


In [40]:
genre_missing_df.columns = ['Всего игр', 'Пропущено оценок', 'Процент пропусков']

In [41]:
genre_missing_df

,Всего игр,Пропущено оценок,Процент пропусков
genre,,,
action,3369,1479,44.0
adventure,1303,980,75.0
fighting,849,440,52.0
misc,1750,1227,70.0
platform,888,391,44.0
puzzle,580,356,61.0
racing,1249,507,41.0
role-playing,1498,761,51.0
shooter,1323,379,29.0


In [42]:
year_missing_df.columns = ['Всего игр', 'Пропущено оценок', 'Процент пропусков']

In [43]:
year_missing_df

,Всего игр,Пропущено оценок,Процент пропусков
year_of_release,,,
1980.0,9,9,100.0
1981.0,46,46,100.0
1982.0,36,36,100.0
1983.0,17,17,100.0
1984.0,14,14,100.0
1985.0,14,13,93.0
1986.0,21,21,100.0
1987.0,16,16,100.0
1988.0,15,14,93.0


**Анализ закономерностей, связанных с пропусками в поле `critic_score`**
- **Связь с популярностью платформ:** Критики чаще оценивают игры для платформ с большим количеством тайтлов. Вероятно, профильные издания и блогеры отдают приоритет более популярным платформам и громким релизам, чтобы привлечь максимальную аудиторию к своим рецензиям.
- **Жизнесопособность платформ:** Платформы вроде `Atari 2600`, `3DO` и `SNES` ушли со сцены до формирования института профессиональной игровой журналистики в том виде, в котором её собирают агрегаторы (Metacritic, GameRankings). Данные по играм 80-х и 90-х в современных датасетах — это результат «ретроспективного» заполнения (когда критики оценивали классику задним числом) или оцифровки старых печатных журналов (EGM, Famitsu, «Великий Дракон»). Именно поэтому оценки за прошлый век носят **единичный характер.**
- **Повсеместность пропусков:** При этом ни на одной платформе нет 100%-й заполненности оценок. Это логично: даже на самых успешных консолях выходит много нишевых, инди- или сугубо локальных проектов (например, ориентированных только на внутренний рынок Японии, США или России), которые не получают внимания со стороны крупных мировых игровых СМИ. Если исходные данные собирались в рамках русскоязычного сегмента интернета, для обогащения датасета и заполнения пропусков целесообразно интегрировать данные из альтернативных международных агрегаторов.
- Реже всего критики оценивают игры жанров `adventure` и `misc`, количество пропусков составляет 75% и 70% соответственно. При этом количество игр представленных категорий значительное на рынке (3369 и 1750). Возможно, именно разнообразие этих продуктов привело к тому, что не все игры заслужили внимание критиков или не вызвали резонанс на рынке, тем самым не были интересны экспертам.
- Как и следовало ожидать, игры, выпущенные в конце **XX века** практически не оценены критиками. Оценки критиков есть только в данных за 1985, 1988, 1992, 1994. Начиная с 1996 года по 2002 мы наблюдаем следующую **тенденцию:** бурный рост количества выпущенных игр за год и числа оценок критиков. Так в данных за 2002 есть только **24%** пропусков. Последующие годы с 2005 по 2009 характеризуется ростом количества игр и возрастающим пропуском в оценках критиков. Можно предположить, что рынок развивался настолько стремильно, что был перенасыщен, поэтому не все игры стали востребоваными и обсуждаемыми в кругах профессионалов. Дальше мы наблюдаем значительный спад количества игр и оценок. Возможно, игровая культура на консоли перешла в **mobile** и **free-to-play**, а производители стали опираться не на жанровое разнообразие, а на качество продукта, улучшая его и выпуская новые версии.

Выявим характер пропусков оценок пользователей в столбце `user_score`

In [44]:
platform_missing_count = cleaned_df.groupby('platform')['is_user_score_missing'].sum()

Количество пропущенных оценок пользователей для каждой платформы.

In [45]:
platform_missing_pct = round(cleaned_df.groupby('platform')['is_user_score_missing'].sum() / cleaned_df.groupby('platform')['platform'].count(),2) * 100

Процент пропущенных оценок пользователей для каждой платформы

In [46]:
platform_missing_user_df = pd.concat([platform_total_count, platform_missing_count, platform_missing_pct], axis=1)

Создаем датафрейм `platform_missing_user_df`, содержащий в себе столбцы с количеством игр на каждой платформе, количеством и процентом пропущенных оценок пользователей для каждой платформы.

In [47]:
genre_missing_count = cleaned_df.groupby('genre')['is_user_score_missing'].sum()

Количество пропущенных оценок пользователей для каждого жанра.

In [48]:
genre_missing_pct = round(cleaned_df.groupby('genre')['is_user_score_missing'].sum() / cleaned_df.groupby('genre')['genre'].count(),2) * 100

Процент пропущенных оценок пользователей для каждого жанра.

In [49]:
genre_missing_user_df = pd.concat([genre_total_count, genre_missing_count, genre_missing_pct], axis=1)

Создаем датафрейм `genre_missing_user_df`, содержащий в себе столбцы с количеством игр каждого жанра, количеством и процентом пропущенных оценок пользователей для каждого жанра.

In [50]:
year_missing_count = cleaned_df.groupby('year_of_release')['is_user_score_missing'].sum()

Количество пропущенных оценок пользователей по каждому году.

In [51]:
year_missing_pct = round(cleaned_df.groupby('year_of_release')['is_user_score_missing'].sum() / cleaned_df.groupby('year_of_release')['year_of_release'].count(),2) * 100

Процент пропущенных оценок пользователей по каждому году.

In [52]:
year_missing_user_df = pd.concat([year_total_count, year_missing_count, year_missing_pct], axis=1)

Создаем датафрейм `year_missing_user_df`, содержащий в себе столбцы с количеством игр для каждого года выпуска, количеством и процентом пропущенных оценок пользователей для каждого года выпуска.

In [53]:
platform_missing_user_df

,platform,is_user_score_missing,0
platform,,,
2600,133,133,100.0
3DO,3,3,100.0
3DS,520,290,56.0
DC,52,38,73.0
DS,2151,879,41.0
GB,98,98,100.0
GBA,822,305,37.0
GC,556,87,16.0
GEN,29,29,100.0


In [54]:
genre_missing_user_df

,genre,is_user_score_missing,0
genre,,,
action,3369,1163,35.0
adventure,1303,844,65.0
fighting,849,411,48.0
misc,1750,862,49.0
platform,888,321,36.0
puzzle,580,241,42.0
racing,1249,375,30.0
role-playing,1498,725,48.0
shooter,1323,300,23.0


In [55]:
year_missing_user_df

,year_of_release,is_user_score_missing,0
year_of_release,,,
1980.0,9,9,100.0
1981.0,46,46,100.0
1982.0,36,36,100.0
1983.0,17,17,100.0
1984.0,14,14,100.0
1985.0,14,13,93.0
1986.0,21,21,100.0
1987.0,16,16,100.0
1988.0,15,14,93.0


**Анализ закономерностей, связанных с пропусками в поле `user_score`**
- **Распределение пропусков по жанрам неоднородно.** Наибольшая доля пропусков в user_score наблюдается в жанре `adventure.` Это может указывать на то, что наличие пользовательской оценки зависит от характеристик самой игры и её аудитории. Одним из возможных объяснений может быть возрастная структура аудитории: игры определённых жанров могли быть ориентированы на более узкие или специфические группы пользователей, которые менее активно оставляли оценки. Однако имеющихся данных недостаточно, чтобы однозначно установить такую причинно-следственную связь.
- **При анализе пропусков по годам наблюдается выраженная временная закономерность.** Для игр ранних периодов доля отсутствующих пользовательских оценок значительно выше. По мере развития игровой индустрии и появления более распространённых онлайн-платформ для публикации пользовательских оценок количество доступных отзывов и оценок увеличивается. При этом общая динамика пропусков в `user_score` во многом сопоставима с динамикой пропусков в `critic_score:` для более ранних игр оценки представлены значительно менее полно.
- **При этом пользовательские оценки в целом представлены несколько полнее, чем оценки критиков.** Это может быть связано с различиями в способе формирования данных: для публикации пользовательской оценки не требуется профессиональная экспертиза, а участие пользователей потенциально доступно более широкому кругу людей. Кроме того, развитие интернет-платформ и игровых сообществ упростило возможность оставлять оценки и отзывы. Однако данная гипотеза не может быть подтверждена только на основании рассматриваемого набора данных.

С учётом высокой доли пропусков и выявленной неоднородности их распределения заполнение отсутствующих значений средним или медианным значением может привести к искажению результатов анализа. Поэтому в дальнейшем целесообразно **сохранить пропуски как NaN** или заменить их на **значение-индикатор** и использовать для расчётов оценки только тех игр, для которых значение `critic_score` и `user_score` действительно известно.

Для корректного изменения типов данных заменим значения NaN в столбце `year_of_release` на значения-индикаторы 

In [56]:
cleaned_df['year_of_release'] = cleaned_df['year_of_release'].fillna(0)

Проверим наличие пропусков в данном столбце

In [57]:
cleaned_df['year_of_release'].isna().sum()

0

Заменим значения NaN в столбцах `critic_score` и `user_score` на значение индикатор 

In [58]:
cleaned_df['critic_score'] = cleaned_df['critic_score'].fillna(-1)

Проверим наличие пропусков в этом столбце

In [59]:
cleaned_df['critic_score'].isna().sum()

0

In [60]:
cleaned_df['user_score'] = cleaned_df['user_score'].fillna(-1)

Проверим наличие пропусков в данном столбце

In [61]:
cleaned_df['user_score'].isna().sum()

0

### Изменение типов данных

На этапе знакомства с данными мы проанализировали соответствие типов столбцов хранимым данным. После заполнения пропусков преобразуем типы.

In [62]:
cleaned_df['eu_sales'] = pd.to_numeric(cleaned_df['eu_sales'], errors='coerce')

In [63]:
cleaned_df['eu_sales'].dtype

dtype('float64')

In [64]:
cleaned_df['jp_sales'] = pd.to_numeric(cleaned_df['jp_sales'], errors='coerce')

In [65]:
cleaned_df['jp_sales'].dtype

dtype('float64')

In [66]:
cleaned_df['user_score'] = pd.to_numeric(cleaned_df['user_score'], errors='coerce')

In [67]:
cleaned_df['user_score'].dtype

dtype('float64')

In [68]:
cleaned_df['critic_score'] = pd.to_numeric(cleaned_df['critic_score'], errors='coerce')

In [69]:
cleaned_df['critic_score'].dtype

dtype('float64')

Меняем `object` на `float64` с помощью метода `to_numeric` и параметра `coerce`. Если тип, указанный в строке нельзя привести к числовому, то такие данные будут замещены Nan и представимы в качестве пропуска

In [70]:
cleaned_df['year_of_release'] = pd.to_numeric(cleaned_df['year_of_release'], errors='coerce')

In [71]:
cleaned_df['year_of_release'].dtype

dtype('float64')

In [72]:
cleaned_df['year_of_release'].isna().sum()

0

Несмотря на то, что тип столбца не изменился, пропуски были заменены на Nan, поэтому теперь к столбцу `year_of_release` можно принимать математические методы.

---

## Фильтрация данных
Отбираем данные за период с 2000 по 2013 год включительно. Сохраняем новый срез данных в отдельном датафрейме `df_actual`.

In [73]:
df_actual = cleaned_df[(cleaned_df['year_of_release'] >= 2000) & cleaned_df['year_of_release'] <= 2013]

In [74]:
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,is_critic_score_missing,is_user_score_missing
0,wii sports,Wii,2006.0,sports,41.36,28.96,3.77,8.45,76.0,8.0,E,False,False
1,super mario bros.,NES,1985.0,platform,29.08,3.58,6.81,0.77,-1.0,-1.0,NaN,True,True
2,mario kart wii,Wii,2008.0,racing,15.68,12.76,3.79,3.29,82.0,8.3,E,False,False
3,wii sports resort,Wii,2009.0,sports,15.61,10.93,3.28,2.95,80.0,8.0,E,False,False
4,pokemon red/pokemon blue,GB,1996.0,role-playing,11.27,8.89,10.22,1.00,-1.0,-1.0,NaN,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16951,samurai warriors: sanada maru,PS3,2016.0,action,0.00,0.00,0.01,0.00,-1.0,-1.0,NaN,True,True
16952,lma manager 2007,X360,2006.0,sports,0.00,0.01,0.00,0.00,-1.0,-1.0,NaN,True,True
16953,haitaka no psychedelica,PSV,2016.0,adventure,0.00,0.00,0.01,0.00,-1.0,-1.0,NaN,True,True
16954,spirits & spells,GBA,2003.0,platform,0.01,0.00,0.00,0.00,-1.0,-1.0,NaN,True,True


---

## Категоризация данных
    
Проведем категоризацию данных:
Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала) и оценкам критиков. Категории для оценок критиков: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала). Для этого используем метод `pd.cut` с параметром `right=False`, таким образом будет включаться верхняя (правая граница). Именно поэтому в качестве последней границы мы указали 11 и 101. 


In [75]:
df_actual['rating_user_score'] = pd.cut(df_actual['user_score'], bins=[-1, 0, 3, 8, 11], right=False, labels=['без категории','низкая оценка', 'средняя оценка', 'высокая оценка'])

In [76]:
df_actual['rating_critic_score'] = pd.cut(df_actual['critic_score'], bins=[-1, 0, 30, 80, 101], right=False, labels=['без категории','низкая оценка', 'средняя оценка', 'высокая оценка'])

Сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [77]:
count_user_category = df_actual.groupby('rating_user_score')['rating_user_score'].count()

In [78]:
count_user_category

rating_user_score
без категории     6701
низкая оценка      145
средняя оценка    4857
высокая оценка    2588
Name: rating_user_score, dtype: int64

In [79]:
count_criric_category = df_actual.groupby('rating_critic_score')['rating_critic_score'].count()

In [80]:
count_criric_category

rating_critic_score
без категории     8578
низкая оценка       62
средняя оценка    6063
высокая оценка    2012
Name: rating_critic_score, dtype: int64

Количество средних оценок, выставленных критиками, более чем в 3 раза превышает количество высоких. В оценках пользователей наблюдается меньший разрыв между количеством средних и высоких. При этом количество игр без оценки значительно превосходит все оцененные игры.

Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [81]:
top_platform = df_actual.groupby('platform')['platform'].count().sort_values(ascending=False)

In [82]:
top_platform.head(7)

platform
PS2     2161
DS      2151
PS3     1331
Wii     1320
X360    1262
PSP     1209
PS      1197
Name: platform, dtype: int64

---

## Итоговый вывод



### Общий обзор проделанной работы
В ходе проекта были выполнены следующие шаги:
* Предобработка и очистка данных: Привели названия столбцов к стилю `snake_case`, привели типы данных к корректным форматам с помощью `to_numeric()` (включая столбец `year_of_release`), устранили неявные дубликаты в категориальных признаках (`platform`, `genre`, `year_of_release`) и удалили 241 явный дубликат (~1% датафрейма), сохранив результат в `cleaned_df`.
* Анализ пропусков: Изучили распределение отсутствующих значений в разрезе платформ, жанров и лет выпуска с помощью флагов `is_critic_score_missing` и `is_user_score_missing`. Выявили, что пропуски в оценках не являются случайными (чаще встречаются у более старых игр), поэтому оставили их без заполнения, как и пропуски в столбцах `name`, `year_of_release` (<2%) и `rating` (>41%), чтобы не искажать исходные данные.
* Фильтрация и категоризация: Сформировали актуальный срез данных за период с 2000 по 2013 год и разделили игры на три категории (низкие, средние и высокие оценки) отдельно по рейтингам критиков (`critic_score`) и пользователей (`user_score`).
* Сравнительный анализ оценок: Выявили заметную разницу в распределении рейтингов — у критиков количество игр со средними оценками более чем в три раза превышает число высокооценённых проектов, тогда как у пользователей разрыв между средними и высокими оценками значительно меньше.


### Исследовательский вывод: анализ закономерностей пропусков в оценках

Проведённый анализ показал, что пропуски в полях `critic_score` и `user_score` имеют системный характер и напрямую отражают эволюцию игровой индустрии, а не случайные ошибки сбора данных:

* **Историческая динамика и развитие индустрии:** Игры конца XX века (1980-е и первая половина 1990-х) практически не имеют оценок, поскольку эпоха устаревших платформ (Atari 2600, 3DO, SNES) пришлась на период до появления крупных онлайн-агрегаторов. С 1996 по 2002 год полнота данных резко выросла (до 76% заполненности оценок критиков в 2002 году), однако на пике рынка в 2005–2009 годах доля пропусков снова увеличилась из-за перенасыщения индустрии релизами. После 2009 года общий спад консольных релизов и оценок связан с перетоком аудитории в сегменты mobile и free-to-play.
* **Влияние популярности платформ и нишевости:** Критики приоритетно оценивают проекты для массовых платформ с большой библиотекой игр. При этом ни одна консоль не имеет 100% заполненности оценок из-за большого числа инди-проектов и игр, ориентированных на локальные рынки (Япония, США, Россия), которые не попадают в фокус мировых СМИ.
* **Жанровая специфика:** Наибольшая доля пропусков как у критиков (75% и 70%), так и у пользователей зафиксирована в категориях `adventure` и `misc`. При высоком общем объёме таких релизов (3369 и 1750 игр соответственно) их высокая фрагментированность и узкая целевая аудитория привели к тому, что большинство проектов не вызвали широкого резонанса.
* **Разница между оценками критиков и пользователей:** Динамика пропусков у обеих групп схожа, но пользовательский рейтинг заполнен полнее. Это объясняется более низким порогом входа для игроков и развитием профильных интернет-сообществ, позволяющих оценивать проекты без профессиональной экспертизы.
* **Методологическое решение:** Из-за высокой доли пропусков и их неоднородного распределения заполнение отсутствующих оценок средними или медианными значениями привело бы к сильному искажению реальной картины. Пропуски оставлены в формате `NaN` (с сохранением флагов-индикаторов), чтобы дальнейшие расчёты строились исключительно на фактически выставленных рейтингах.
